# PROVE BEFORE TRADE — Paper Trading + Feature Lab
**เงินจริงยังไม่ถูกใช้** ระบบนี้ใช้ข้อมูลจริงเพื่อทดสอบหลักฐาน และใช้เงินจำลอง 1,000 บาทสำหรับ Paper Trading


In [ ]:
!pip -q install requests==2.32.5 pandas==2.2.3 "numpy>=1.26,<3" "yfinance>=0.2.54,<1"


In [ ]:
import os
REPO='https://github.com/gujibata2-dotcom/adaptive-market-intelligence.git'
BRANCH='feat/paper-trading-bot'
if not os.path.exists('/content/adaptive-market-intelligence'):
    !git clone -b $BRANCH $REPO /content/adaptive-market-intelligence
%cd /content/adaptive-market-intelligence/paper_bot
!python test_core.py
!python test_proof_gate.py


## 1) ตรวจราคาจริง BTC/THB จาก Bitkub


In [ ]:
from bot.market import get_last_price
price = get_last_price('BTC_THB')
print('BTC/THB:', price)


## 2) Feature Lab — ให้ข้อมูลคัดตัวแปรเอง


In [ ]:
from bot.feature_lab import run_feature_lab
scores = run_feature_lab()
display(scores)


## 3) Live Gate — ตรวจว่าหลักฐานถึงเกณฑ์หรือยัง


In [ ]:
from bot.proof_gate import evaluate_live_gate
current_metrics = {
    'paper_days': 0,
    'closed_trades': 0,
    'net_return_pct': 0,
    'max_drawdown_pct': 0,
    'profit_factor': 0,
    'out_of_sample_pass': False,
    'walk_forward_pass': False,
    'fee_slippage_stress_pass': False,
    'regimes_tested': 0,
    'regime_robustness_pass': False,
    'prediction_ledger_locked': False,
}
gate = evaluate_live_gate(current_metrics)
gate


## 4) Paper Trading — ทดลองระบบซื้อขายด้วยเงินจำลอง


In [ ]:
from bot.engine import TradingEngine
engine = TradingEngine(symbol='BTC_THB', start_thb=1000, fast=5, slow=20, log_path='/content/trading_log.csv')
print('PAPER ENGINE READY — NO REAL ORDERS')


In [ ]:
import time, pandas as pd
for i in range(25):
    try:
        result = engine.step()
        print(i+1, result)
    except Exception as e:
        print('ERROR:', e)
    time.sleep(10)


In [ ]:
df = pd.read_csv('/content/trading_log.csv')
display(df.tail(30))
print('Start equity: 1000 THB')
print('Latest equity:', df.iloc[-1]['equity'] if len(df) else 'no data')
